# Подключение и импорты

In [ ]:
import json
import sys

import numpy as np
import pandas as pd
from sqlalchemy import text

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            # Моменты времени считаем в UTC: так они у всех одинаковые.
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


print("подключились как:", q("select current_user as login").iloc[0, 0])

Выручка - основной показатель бизнеса, поэтому посмотрим на его динамику по неделям

In [ ]:
payments = q(f"""
    select date_trunc('week', paid_at) as week
        , count(*) as payments_cnt
        , sum(amount) as revenue
    from prime.payments
    where status = 'success'
    group by 1 
    order by 1;
""")

In [ ]:
# ---------- подготовка ----------
def prep(df, lc=None):
    df = df.copy()
    df["week"] = pd.to_datetime(df["week"]).dt.tz_localize(lc)
    return df

payments_ = prep(payments)

payments_["revenue"] = payments_["revenue"].astype(float)

payments_["avg_check"] = payments_["revenue"] / payments_["payments_cnt"]

fig, axes = plt.subplots(figsize=(12, 5))
axes.plot(payments_["week"], payments_["revenue"] / 1e6)
axes.set_xlabel("Дата")
axes.set_ylabel("Выручка, млн руб")
axes.set_title("Выручка по неделям")

plt.tight_layout()
plt.show()

На данном графике есть сразу 3 интересных места:
1. Резкий рост где-то в феврале 2026
2. Резкий провал в апреле 2026
3. Резкое падение в августе-сентябре 2026

Чем может быть вызвано каждое из них:
1. Поднятие цен/маркетинговая кампаний
2. Ошибка в данных/данные не доехали
3. Не доехали данные, так как это конец рассматриваемого периода

Я остановлюсь на первой гипотезе и рассмотрю ее более подробно

# Гипотеза: рост выручки на стыке февраля и марта 2026

**Описание.** На графике выручки виден явный рост где-то в феврале 2026 года, и достигнутый уровень не исчез со временем

**Бизнес-вопрос.** За счёт чего выросла выручка от подписок в конце февраля-начале марта 2026 и что с этим делать: закреплять новые условия, откатывать их или повторять приём?

**Гипотеза.** Рост вызван одномоментным повышением цен на тарифы, а не ростом числа платящих клиентов

## Метрики и зерно

**Мультипликативная модель:** `выручка = число платежей × средний чек`

**Зерно: один успешный платёж, агрегированный до «день × тариф».**
- Платёж, а не клиент и не подписка: выручку создаёт именно списание, поэтому суммируем строки `payments` со `status = 'success'`.
- Тариф берём из `subscriptions` по `subscription_id`, часовой пояс — из `clients` по `owner_client_id`. Оба соединения «многие к одному» (у платежа одна подписка, у подписки один владелец) и строки не размножают; это проверяется сверкой с контрольным запросом без соединений. **Таблицу `subscription_members` не трогаем**: платит владелец, участники шеринга выручку не создают.
- День, а не неделя: чтобы увидеть, ступенька это (изменение условий) или всплеск в отдельные дни
- Фактическая цена — это `payments.amount`. Справочник `tariffs` нужен, чтобы проверить, что его цены совпадают со списаниями, и найти дату смены цены.
- Окна сравнения: **4 недели до** и **4 недели с начала скачка**

## Шаг 1. Справочник цен: есть ли в нём дефекты

Прежде чем сопоставлять платежи со справочником `tariffs`, проверяем сам справочник. В нём хранится **история цен**: у одного тарифа несколько строк с интервалами действия. Если интервалы двух строк пересекаются, то платёж найдёт в справочнике *две* подходящие строки, и после соединения платежей станет больше, чем есть на самом деле.

In [ ]:
# 1. Справочник цен: нет ли пересекающихся интервалов и менялась ли цена в нужную дату
tariffs = q("select * from prime.tariffs order by tariff_id, valid_from")
# Даты справочника — дни без часового пояса, а платежи — моменты в UTC.
ref = tariffs.copy()
for col in ("valid_from", "valid_to"):
    ref[col] = pd.to_datetime(ref[col]).dt.tz_localize("UTC")

# 1. Справочник без строки, из-за которой интервалы одного тарифа перекрываются.
a = tariffs.reset_index(names="row_id")
pairs = a.merge(a, on="tariff_id", suffixes=("_a", "_b"))

overlap_mask = (
    (pairs["row_id_a"] < pairs["row_id_b"])              # без этого условия каждая
    & (pairs["valid_from_a"] <= pairs["valid_to_b"])      # строка пересечётся сама
    & (pairs["valid_from_b"] <= pairs["valid_to_a"])      # с собой
)
in_overlaps = pd.concat([pairs.loc[overlap_mask, "row_id_a"], pairs.loc[overlap_mask, "row_id_b"]]).value_counts()
bad_row = in_overlaps.index[0]
ref_fixed = ref.drop(index=bad_row)
ref_fixed

## Шаг 2. Выгрузка платежей и часовые пояса

**Зерно выгрузки — один успешный платёж** (`payment_id`). Тариф берём из `subscriptions` по `subscription_id`, часовой пояс — из `clients` по `owner_client_id` (платит владелец). Оба соединения «многие к одному», поэтому строки не размножаются; это проверяется сверкой с контрольным запросом по `payments` без соединений. `subscription_members` не нужна: участники шеринга выручку не создают, а присоединение к ним дало бы двойной счёт.

**Часовые пояса.** В базе всё хранится в UTC, а клиенты живут в нескольких поясах, поэтому «день по UTC» и «день клиента» отличаются

In [ ]:
payments = q(f"""
with payments as (
                    select *
                    from prime.payments
                    where status = 'success'
                    and paid_at >= '2026-02-01'
                    and paid_at < '2026-04-01'
                )
    , subscriptions as (
                        select *
                        from prime.subscriptions
                    )
    , clients as (
                    select client_id
                        , time_zone
                    from prime.clients
                )
select payment_id
    , s.subscription_id
    , paid_at
    , amount
    , payment_method
    , owner_client_id
    , tariff_id
    , started_at
    , ended_at
    , is_trial
    , time_zone
from payments p 
left join subscriptions s on p.subscription_id = s.subscription_id
left join clients c on s.owner_client_id = c.client_id
""")

payments

In [ ]:
for time_col in ['paid_at', 'started_at', 'ended_at']:
    payments[time_col] = pd.to_datetime(payments[time_col], utc=True)
    payments[f'{time_col}_local'] = payments.apply(
        lambda row: row[time_col].tz_convert(row['time_zone']),
        axis=1
    )

payments['paid_at_local_day'] = payments['paid_at_local'].apply(lambda x: x.date())
payments.head()

## Шаг 3. Справочник против фактических списаний

Соединяем платежи со справочником по `tariff_id` и попаданию `paid_at` в интервал.

**Конвенция курса:** `paid_at >= valid_from` и `paid_at < valid_to + 1 день`. Даты справочника приводим к UTC, потому что все `paid_at` хранятся в UTC.

In [ ]:
payments_tariff = (
    payments.merge(ref_fixed, how='left', on='tariff_id')
    .loc[lambda df:
        (df['paid_at'] >= df['valid_from']) &
        (df['paid_at'] < df['valid_to'] + pd.Timedelta(days=1))
    ]
)
payments_tariff.head()

## Шаг 4. Графики: декомпозиция выручки

Агрегируем платежи до **«локальный день × тариф»**. Это зерно нужно, чтобы отличить ступеньку (изменение условий с конкретной даты) от всплеска в отдельные дни: недельная агрегация размыла бы момент излома

Сверху число платежей по тарифам, снизу средний чек. Если причина в цене, то на нижнем графике средний чек сдвинется ступенькой около даты смены цены, а число платежей сверху останется на прежнем уровне. Если причина в кампании, то будет наоборот.

In [ ]:
payments_agg = payments.groupby('paid_at_local_day', as_index=False).agg(payments_cnt=('payment_id', 'count'), revenue=('amount', 'sum'))
payments_agg["revenue"] = payments_agg["revenue"].astype(float)
payments_agg["avg_check"] = payments_agg["revenue"] / payments_agg["payments_cnt"]
payments_agg.head()

fig, axes = plt.subplots(figsize=(12, 4))
axes.plot(payments_agg["paid_at_local_day"], payments_agg["revenue"] / 1e6, marker="o")
axes.set_xlabel("Дата")
axes.set_ylabel("Выручка, млн руб")
axes.set_title("Выручка по дням")

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)

axes[0].plot(payments_agg["paid_at_local_day"], payments_agg["payments_cnt"] / 1e3, marker="o")
axes[0].set_title("Успешных платежей по дням")
axes[0].set_ylabel("Успешных платежей, тыс.")

axes[1].plot(payments_agg["paid_at_local_day"], payments_agg["avg_check"], marker="o")
axes[1].set_title("Средний чек по дням")
axes[1].set_ylabel("Средний чек, ₽")
plt.tight_layout()
plt.show()

## Шаг 5. График: динамика оплат в разрезе тарифов

Ожидаем увидеть, что в определенный момент происходит смена тарифов -> повышение цены -> повышение выручки

In [ ]:
import seaborn as sns

payments_agg_tariff = payments_tariff.groupby(['paid_at_local_day', 'monthly_fee'], as_index=False).agg(payment_cnt=('payment_id', 'count'))
payments_agg_tariff.head()

plt.figure(figsize=(16, 6))

sns.lineplot(
    data=payments_agg_tariff,
    x='paid_at_local_day',
    y='payment_cnt',
    hue='monthly_fee',
    markers=False,
    palette='muted'
)

plt.title('Динамика количества оплат по дням')
plt.xlabel('Дата')
plt.ylabel('Количество оплат')
plt.legend(title='Тариф / цена')
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Вывод и рекомендация

**Бизнес-вопрос:** за счёт чего выросла выручка в начале марта 2026 и что с этим делать.

## Что показали данные

1. **Причина роста — повышение цен.** По справочнику `tariffs` с 01.03.2026 цены выросли: «Базовый» 199 → 249 ₽ (+25 %), «Расширенный» 399 → 449 ₽ (+12,5 %). Средний чек ступенькой вырос примерно с 275 до 325 ₽ (около +18 %) в дату смены цены, а число платежей в день осталось почти прежним.

2. **Рост закрепился.** После скачка уровень выручки держится на новом уровне весь остальной период (провал апреля — отдельный эпизод, который мы не разбираем).

## Рекомендации

1. **Оставить новые цены и не откатывать их.** Повышение дало основную часть роста выручки без потери объёма платежей.

4. **Исправить справочник цен.** В `tariffs` есть строка тарифа «Расширенный», интервал которой пересекается с соседними. Любое соединение платежей со справочником «по интервалу» без этой проверки размножает платежи; нужна валидация справочника на пересечения при загрузке.

## Ограничения

- Это наблюдательный анализ без контрольной группы: причинность подтверждается совпадением даты ступеньки со сменой цены в справочнике и согласованностью с ожидаемым эффектом, а не экспериментом.

- Падение в апреле и последняя неделя ряда не исследовались.

- Календарные дни считаются по поясу владельца подписки, а границы окон сравнения — по UTC; недельный ряд в начале ноутбука — по UTC.